In [ ]:
!pip install gdown

In [ ]:
!gdown --folder https://drive.google.com/drive/folders/1lm5JrRj-v2KjwTnqalWWDggNer5Zkd8Y?usp=sharing

Retrieving folder contents
Processing file 1HgGkoCkgQIHm9RBLHVDk2Ymffj114-51 baseline.ipynb
Processing file 1BluU1LL2Vbp3r7j6xxdePSVrNK1bQ3Ne customers.csv
Processing file 17GPgEUQtEuNJq336zdrRba53he-Pzqm8 geography.csv
Processing file 18TAFcPhelq3zd2hhCeJ2G4TVyikQZhCh inventory.csv
Processing file 1WVZ-qo1PXT0zX5p8cn87sY8jnT7Rs0Ck order_items.csv
Processing file 1bqG6Y1NsbC2JV5ODrUWhvQ4F8LUF0xUu orders.csv
Processing file 1cf5XgkY2Y141meQ0Maqwogviqkqk99ba payments.csv
Processing file 1VSRvIsRgJg5doz--KKpZONlJBAsppdVZ products.csv
Processing file 1Z2LMXiHdbDiQR8TqgTT86QHeamVECNYz promotions.csv
Processing file 1jHTTm7wdkj8fPEwRi5UmmL2qt3xfNPrj returns.csv
Processing file 1c9SvMMJV2BjN2LLl6A-z_R-IYqMHHJCQ reviews.csv
Processing file 1Iv9fhNMifdmx7E_SRzhq6ylut9UuitOx sales.csv
Processing file 1QVlkk8Co4GCwgnK8DJccUcNI9_bIPC2m sample_submission.csv
Processing file 1GC8PSUkoEU-Vm5cVxrge0OZLZmP2xE0d shipments.csv
Processing file 1JJVMpQqSSjzKnzf1RfRRQeRWBrKsn6TY web_traffic.csv
Retrieving f

In [ ]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 7.6 MB/s eta 0:00:00


In [ ]:
# ============================================================
# DATATHON 2026 — Revenue Forecasting v5b
# Same as v5 but replaces CatBoost with:
#   - sklearn HistGradientBoostingRegressor (fast, robust)
#   - sklearn ExtraTreesRegressor (high variance diversity)
# No extra installs needed beyond xgboost + lightgbm
# ============================================================

import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import warnings
warnings.filterwarnings("ignore")

import random, os
SEED = 42
random.seed(SEED); np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ============================================================
# 1. LOAD DATA
# ============================================================
print("=" * 60)
print("1. LOADING DATA")
print("=" * 60)

sales       = pd.read_csv("Data/sales.csv")
promotions  = pd.read_csv("Data/promotions.csv")
web_traffic = pd.read_csv("Data/web_traffic.csv")
inventory   = pd.read_csv("Data/inventory.csv")
customers   = pd.read_csv("Data/customers.csv")
sample_sub  = pd.read_csv("Data/sample_submission.csv")

for df, col in [(sales, "Date"), (sample_sub, "Date")]:
    df[col] = pd.to_datetime(df[col])
web_traffic["date"]        = pd.to_datetime(web_traffic["date"])
inventory["snapshot_date"] = pd.to_datetime(inventory["snapshot_date"])
promotions["start_date"]   = pd.to_datetime(promotions["start_date"])
promotions["end_date"]     = pd.to_datetime(promotions["end_date"])
customers["signup_date"]   = pd.to_datetime(customers["signup_date"])

sales = sales.sort_values("Date").reset_index(drop=True)
print(f"  Train: {sales['Date'].min().date()} → {sales['Date'].max().date()} ({len(sales)} rows)")
print(f"  Test : {sample_sub['Date'].min().date()} → {sample_sub['Date'].max().date()} ({len(sample_sub)} rows)")

# ============================================================
# 2. STATIC FEATURE ENGINEERING
# ============================================================
print("\n" + "=" * 60)
print("2. STATIC FEATURE ENGINEERING")
print("=" * 60)

ALL_DATES = pd.date_range(sales["Date"].min(), sample_sub["Date"].max(), freq="D")
base = pd.DataFrame({"Date": ALL_DATES})
base = base.merge(sales[["Date", "Revenue", "COGS"]], on="Date", how="left")
base = base.sort_values("Date").reset_index(drop=True)

d = base["Date"]

# ── Calendar ──────────────────────────────────────────────────
base["year"]           = d.dt.year
base["month"]          = d.dt.month
base["day"]            = d.dt.day
base["dayofweek"]      = d.dt.dayofweek
base["dayofyear"]      = d.dt.dayofyear
base["weekofyear"]     = d.dt.isocalendar().week.astype(int)
base["quarter"]        = d.dt.quarter
base["is_weekend"]     = (d.dt.dayofweek >= 5).astype(int)
base["is_month_start"] = d.dt.is_month_start.astype(int)
base["is_month_end"]   = d.dt.is_month_end.astype(int)
base["days_in_month"]  = d.dt.days_in_month
base["dom_early"]      = (d.dt.day <= 10).astype(int)
base["dom_mid"]        = ((d.dt.day > 10) & (d.dt.day <= 20)).astype(int)
base["dom_late"]       = (d.dt.day > 20).astype(int)
base["week_of_month"]  = (d.dt.day - 1) // 7 + 1
base["payday"]         = d.dt.day.isin([1, 15]).astype(int)

# ── Fourier terms ─────────────────────────────────────────────
for k in [7, 14, 30, 60, 91, 182, 365]:
    base[f"sin_{k}"] = np.sin(2 * np.pi * d.dt.dayofyear / k)
    base[f"cos_{k}"] = np.cos(2 * np.pi * d.dt.dayofyear / k)

# ── Seasonality dummies ───────────────────────────────────────
base["tet_window"]     = ((d.dt.month == 1) & (d.dt.day >= 20) |
                          (d.dt.month == 2) & (d.dt.day <= 10)).astype(int)
base["q2_peak"]        = d.dt.month.isin([4, 5, 6]).astype(int)
base["q4_trough"]      = d.dt.month.isin([11, 12]).astype(int)
base["liberation_day"] = ((d.dt.month == 4) & (d.dt.day.between(28, 30))).astype(int)
base["national_day"]   = ((d.dt.month == 9) & (d.dt.day.between(1, 3))).astype(int)
base["labor_day"]      = ((d.dt.month == 5) & (d.dt.day == 1)).astype(int)
base["year_end_sale"]  = ((d.dt.month == 12) & (d.dt.day >= 20)).astype(int)
base["mid_year_sale"]  = ((d.dt.month == 6) & (d.dt.day.between(15, 30))).astype(int)
base["flash_1111"]     = ((d.dt.month == 11) & (d.dt.day == 11)).astype(int)
base["flash_1212"]     = ((d.dt.month == 12) & (d.dt.day == 12)).astype(int)
base["pre_1111"]       = ((d.dt.month == 11) & (d.dt.day.between(8, 10))).astype(int)
base["post_1111"]      = ((d.dt.month == 11) & (d.dt.day.between(12, 14))).astype(int)
base["pre_1212"]       = ((d.dt.month == 12) & (d.dt.day.between(9, 11))).astype(int)
base["post_1212"]      = ((d.dt.month == 12) & (d.dt.day.between(13, 15))).astype(int)

# ── Seasonality indices ───────────────────────────────────────
train_hist = sales.copy()
train_hist["year"]       = train_hist["Date"].dt.year
train_hist["month"]      = train_hist["Date"].dt.month
train_hist["quarter"]    = train_hist["Date"].dt.quarter
train_hist["dayofweek"]  = train_hist["Date"].dt.dayofweek
train_hist["weekofyear"] = train_hist["Date"].dt.isocalendar().week.astype(int)

monthly_idx   = (train_hist.groupby("month")["Revenue"].mean() /
                 train_hist["Revenue"].mean()).rename("monthly_season_idx")
quarterly_idx = (train_hist.groupby("quarter")["Revenue"].mean() /
                 train_hist["Revenue"].mean()).rename("quarterly_season_idx")
dow_idx       = (train_hist.groupby("dayofweek")["Revenue"].mean() /
                 train_hist["Revenue"].mean()).rename("dow_season_idx")
week_idx      = (train_hist.groupby("weekofyear")["Revenue"].mean() /
                 train_hist["Revenue"].mean()).rename("week_season_idx")
mdow = train_hist.groupby(["month","dayofweek"])["Revenue"].mean().reset_index()
mdow["mdow_idx"] = mdow["Revenue"] / train_hist["Revenue"].mean()
mdow = mdow[["month","dayofweek","mdow_idx"]]

base = base.merge(monthly_idx,   on="month",     how="left")
base = base.merge(quarterly_idx, on="quarter",   how="left")
base = base.merge(dow_idx,       on="dayofweek", how="left")
base = base.merge(week_idx,      on="weekofyear",how="left")
base = base.merge(mdow,          on=["month","dayofweek"], how="left")

# ── Promotions ────────────────────────────────────────────────
print("  Building promotions features...")
promo_rows = []
for _, row in promotions.iterrows():
    for d_ in pd.date_range(row["start_date"], row["end_date"], freq="D"):
        promo_rows.append({
            "Date":           d_,
            "discount_value": row["discount_value"],
            "stackable":      row["stackable_flag"],
            "is_pct":         int(row["promo_type"] == "percentage"),
        })
promo_daily = (pd.DataFrame(promo_rows)
    .groupby("Date").agg(
        n_active_promos = ("discount_value", "count"),
        max_discount    = ("discount_value", "max"),
        avg_discount    = ("discount_value", "mean"),
        has_stackable   = ("stackable", "max"),
        has_pct_promo   = ("is_pct", "max"),
    ).reset_index())
promo_daily["promo_flag"] = 1
promo_daily["Date"] = pd.to_datetime(promo_daily["Date"])
base = base.merge(promo_daily, on="Date", how="left")
for c in ["promo_flag","n_active_promos","max_discount",
          "avg_discount","has_stackable","has_pct_promo"]:
    base[c] = base[c].fillna(0)

base["promo_lag1"]       = base["promo_flag"].shift(1).fillna(0)
base["promo_lag7"]       = base["promo_flag"].shift(7).fillna(0)
base["promo_lag14"]      = base["promo_flag"].shift(14).fillna(0)
base["promo_density_14"] = base["promo_flag"].shift(1).rolling(14, min_periods=1).mean()
base["promo_density_30"] = base["promo_flag"].shift(1).rolling(30, min_periods=1).mean()
no_promo = (base["promo_flag"] == 0).astype(int)
base["days_since_promo"] = no_promo.groupby((base["promo_flag"] != 0).cumsum()).cumcount()
# Days until next promo (forward-looking — promo calendar known, no leakage)
base["days_to_promo"]    = (base["promo_flag"][::-1]
                            .shift(1).fillna(0)[::-1]
                            .groupby((base["promo_flag"][::-1] != 0).cumsum()[::-1])
                            .cumcount())

# ── Web traffic ───────────────────────────────────────────────
print("  Building web traffic features (lagged)...")
web_daily = (web_traffic.groupby("date").agg(
    sessions        = ("sessions", "sum"),
    unique_visitors = ("unique_visitors", "sum"),
    page_views      = ("page_views", "sum"),
    avg_bounce      = ("bounce_rate", "mean"),
    avg_dur         = ("avg_session_duration_sec", "mean"),
).reset_index().rename(columns={"date": "Date"}))
web_daily["Date"] = pd.to_datetime(web_daily["Date"])
web_daily["pages_per_session"] = web_daily["page_views"] / (web_daily["sessions"] + 1)
web_daily["engagement_score"]  = (web_daily["avg_dur"] / 60) * (1 - web_daily["avg_bounce"])
base = base.merge(web_daily, on="Date", how="left")
for col in ["sessions","unique_visitors","page_views",
            "avg_bounce","avg_dur","pages_per_session","engagement_score"]:
    base[f"{col}_lag1"]   = base[col].shift(1)
    base[f"{col}_roll7"]  = base[col].shift(1).rolling(7,  min_periods=1).mean()
    base[f"{col}_roll30"] = base[col].shift(1).rolling(30, min_periods=1).mean()
    base = base.drop(columns=[col])

# ── Inventory ─────────────────────────────────────────────────
print("  Building inventory features (lagged monthly)...")
inv_monthly = (inventory.groupby("snapshot_date").agg(
    avg_fill_rate    = ("fill_rate", "mean"),
    avg_sell_through = ("sell_through_rate", "mean"),
    pct_stockout     = ("stockout_flag", "mean"),
    pct_overstock    = ("overstock_flag", "mean"),
    avg_days_supply  = ("days_of_supply", "mean"),
).reset_index().rename(columns={"snapshot_date": "Date"}))
inv_daily = (inv_monthly.set_index("Date")
    .reindex(ALL_DATES).ffill()
    .shift(1).reset_index().rename(columns={"index": "Date"}))
inv_daily["Date"] = pd.to_datetime(inv_daily["Date"])
base = base.merge(inv_daily, on="Date", how="left")

# ── New customers ─────────────────────────────────────────────
new_cust = (customers.groupby("signup_date").size()
    .reset_index(name="new_customers")
    .rename(columns={"signup_date": "Date"}))
new_cust["Date"] = pd.to_datetime(new_cust["Date"])
base = base.merge(new_cust, on="Date", how="left")
base["new_customers"]   = base["new_customers"].fillna(0)
base["new_cust_lag1"]   = base["new_customers"].shift(1)
base["new_cust_roll7"]  = base["new_customers"].shift(1).rolling(7,  min_periods=1).mean()
base["new_cust_roll30"] = base["new_customers"].shift(1).rolling(30, min_periods=1).mean()
base = base.drop(columns=["new_customers"])

# ── Regime ────────────────────────────────────────────────────
base["post_2019"]         = (base["year"] >= 2019).astype(int)
base["post_2019_trend"]   = np.where(base["year"] >= 2019, base["year"] - 2019, 0)
base["years_since_start"] = base["year"] - base["year"].min()

# ── Target encoding ───────────────────────────────────────────
enc_base    = base[base["Date"] < "2022-01-01"][["month","quarter","Revenue"]].dropna()
month_enc   = enc_base.groupby("month")["Revenue"].mean().rename("month_rev_enc")
quarter_enc = enc_base.groupby("quarter")["Revenue"].mean().rename("quarter_rev_enc")
base = base.merge(month_enc,   on="month",   how="left")
base = base.merge(quarter_enc, on="quarter", how="left")

base = base.reset_index(drop=True)
print(f"  Static features built: {base.shape[1]} columns")

# ============================================================
# 3. LAG FEATURE FUNCTION
# ============================================================
LAGS      = [1, 2, 3, 5, 7, 10, 14, 21, 28, 30, 45, 60, 90, 120, 180, 364, 365, 366, 730]
WINDOWS   = [7, 14, 21, 30, 60, 90, 180, 365]
EWM_SPANS = [3, 7, 14, 30, 90]

def add_lag_features(df, rev_series, cogs_series):
    rev_s  = pd.Series(
        rev_series.values  if hasattr(rev_series,  "values") else rev_series,
        index=range(len(df)))
    cogs_s = pd.Series(
        cogs_series.values if hasattr(cogs_series, "values") else cogs_series,
        index=range(len(df)))

    for lag in LAGS:
        df[f"rev_lag_{lag}"]  = rev_s.shift(lag).values
        df[f"cogs_lag_{lag}"] = cogs_s.shift(lag).values

    base_rev = rev_s.shift(1)
    for w in WINDOWS:
        df[f"rev_roll_mean_{w}"]   = base_rev.rolling(w, min_periods=1).mean().values
        df[f"rev_roll_std_{w}"]    = base_rev.rolling(w, min_periods=1).std().values
        df[f"rev_roll_max_{w}"]    = base_rev.rolling(w, min_periods=1).max().values
        df[f"rev_roll_min_{w}"]    = base_rev.rolling(w, min_periods=1).min().values
        df[f"rev_roll_median_{w}"] = base_rev.rolling(w, min_periods=1).median().values

    for span in EWM_SPANS:
        df[f"rev_ewm_{span}"] = base_rev.ewm(span=span, min_periods=1).mean().values

    # Momentum ratios
    df["trend_3_7"]    = df["rev_ewm_3"]        / (df["rev_ewm_7"]          + 1)
    df["trend_7_30"]   = df["rev_roll_mean_7"]  / (df["rev_roll_mean_30"]   + 1)
    df["trend_30_90"]  = df["rev_roll_mean_30"] / (df["rev_roll_mean_90"]   + 1)
    df["trend_90_365"] = df["rev_roll_mean_90"] / (df["rev_roll_mean_365"]  + 1)
    df["trend_ewm"]    = df["rev_ewm_7"]        / (df["rev_ewm_30"]         + 1)

    # YoY comparisons
    df["rev_same_week_ly"]  = rev_s.shift(364).values
    df["rev_same_month_ly"] = rev_s.shift(365).values
    df["rev_ly2"]           = rev_s.shift(730).values
    df["yoy_ratio_7"]  = (df["rev_roll_mean_7"]  /
                          (rev_s.shift(365).rolling(7,  min_periods=1).mean().values + 1))
    df["yoy_ratio_30"] = (df["rev_roll_mean_30"] /
                          (rev_s.shift(365).rolling(30, min_periods=1).mean().values + 1))

    # COGS features
    df["gross_profit_lag1"] = df["rev_lag_1"] - df["cogs_lag_1"]
    df["gross_margin_lag1"] = df["gross_profit_lag1"] / (df["rev_lag_1"] + 1)
    df["cogs_ratio_lag1"]   = df["cogs_lag_1"] / (df["rev_lag_1"] + 1)

    # Volatility
    df["rev_cv_30"] = df["rev_roll_std_30"] / (df["rev_roll_mean_30"] + 1)
    df["rev_cv_90"] = df["rev_roll_std_90"] / (df["rev_roll_mean_90"] + 1)

    # WoW change
    df["wow_change"] = (df["rev_roll_mean_7"] -
                        rev_s.shift(8).rolling(7, min_periods=1).mean().values)
    return df

print("  Building lag/rolling features...")
full_df = add_lag_features(base.copy(), base["Revenue"].fillna(0), base["COGS"].fillna(0))

# ── Interaction features ──────────────────────────────────────
full_df["sessions_x_promo"]   = full_df["sessions_lag1"]   * full_df["promo_flag"]
full_df["traffic_x_discount"] = full_df["sessions_lag1"]   * full_df["max_discount"]
full_df["visitors_x_promo"]   = full_df["unique_visitors_lag1"] * full_df["promo_flag"]

print(f"  Total features: {full_df.shape[1]} columns")

# ============================================================
# 4. SPLIT
# ============================================================
print("\n" + "=" * 60)
print("4. TRAIN / VAL / TEST SPLIT")
print("=" * 60)

TARGET    = "Revenue"
COGS_TGT  = "COGS"
DROP_COLS = ["Date", "Revenue", "COGS"]
FEATURES  = [c for c in full_df.columns if c not in DROP_COLS]

train_df = full_df[full_df["Date"] <  "2022-01-01"].dropna(subset=[TARGET])
val_df   = full_df[(full_df["Date"] >= "2022-01-01") &
                   (full_df["Date"] <= "2022-12-31")].dropna(subset=[TARGET])
test_df  = full_df[full_df["Date"] >  "2022-12-31"]

X_train      = train_df[FEATURES].fillna(0)
y_train      = train_df[TARGET]
y_train_cogs = train_df[COGS_TGT]
X_val        = val_df[FEATURES].fillna(0)
y_val        = val_df[TARGET]
y_val_cogs   = val_df[COGS_TGT]
X_trainval      = pd.concat([X_train, X_val])
y_trainval      = pd.concat([y_train, y_val])
y_trainval_cogs = pd.concat([y_train_cogs, y_val_cogs])

print(f"  Train   : {train_df['Date'].min().date()} → {train_df['Date'].max().date()} ({len(train_df)} rows)")
print(f"  Val     : {val_df['Date'].min().date()} → {val_df['Date'].max().date()} ({len(val_df)} rows)")
print(f"  Test    : {test_df['Date'].min().date()} → {test_df['Date'].max().date()} ({len(test_df)} rows)")
print(f"  Features: {len(FEATURES)}")

# ============================================================
# 5. CV HELPERS
# ============================================================
CV_FOLDS = [
    ("2020-07-01", "2020-12-31"),
    ("2021-01-01", "2021-06-30"),
    ("2021-07-01", "2021-12-31"),
    ("2022-01-01", "2022-06-30"),
    ("2022-07-01", "2022-12-31"),
]

def cv_mae_xgb(params, target=TARGET):
    maes = []
    for fold_start, fold_end in CV_FOLDS:
        ft = full_df[(full_df["Date"] < fold_start) & full_df[target].notna()]
        fv = full_df[(full_df["Date"] >= fold_start) &
                     (full_df["Date"] <= fold_end) & full_df[target].notna()]
        if len(ft) < 100 or len(fv) < 30:
            continue
        m = xgb.XGBRegressor(**params, random_state=SEED, n_jobs=-1,
                              verbosity=0, eval_metric="mae", early_stopping_rounds=40)
        m.fit(ft[FEATURES].fillna(0), ft[target],
              eval_set=[(fv[FEATURES].fillna(0), fv[target])], verbose=False)
        pred = np.clip(m.predict(fv[FEATURES].fillna(0)), 0, None)
        maes.append(mean_absolute_error(fv[target], pred))
    return np.mean(maes) if maes else 1e9

def cv_mae_lgb(params, target=TARGET):
    maes = []
    for fold_start, fold_end in CV_FOLDS:
        ft = full_df[(full_df["Date"] < fold_start) & full_df[target].notna()]
        fv = full_df[(full_df["Date"] >= fold_start) &
                     (full_df["Date"] <= fold_end) & full_df[target].notna()]
        if len(ft) < 100 or len(fv) < 30:
            continue
        m = lgb.LGBMRegressor(**params, random_state=SEED, n_jobs=-1, verbose=-1)
        m.fit(ft[FEATURES].fillna(0), ft[target],
              eval_set=[(fv[FEATURES].fillna(0), fv[target])],
              callbacks=[lgb.early_stopping(40, verbose=False), lgb.log_evaluation(-1)])
        pred = np.clip(m.predict(fv[FEATURES].fillna(0)), 0, None)
        maes.append(mean_absolute_error(fv[target], pred))
    return np.mean(maes) if maes else 1e9

# ============================================================
# 6. OPTUNA — XGB
# ============================================================
print("\n" + "=" * 60)
print("6. OPTUNA — XGB REVENUE")
print("=" * 60)

def objective_xgb(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 800, 5000, step=100),
        "learning_rate":     trial.suggest_float("learning_rate", 0.003, 0.04, log=True),
        "max_depth":         trial.suggest_int("max_depth", 3, 7),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.4, 1.0),
        "colsample_bynode":  trial.suggest_float("colsample_bynode",  0.4, 1.0),
        "min_child_weight":  trial.suggest_int("min_child_weight", 5, 60),
        "reg_alpha":         trial.suggest_float("reg_alpha",  0.01, 20.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 0.05, 20.0, log=True),
        "gamma":             trial.suggest_float("gamma", 0.0, 3.0),
        "grow_policy":       trial.suggest_categorical("grow_policy", ["depthwise","lossguide"]),
    }
    return cv_mae_xgb(params)

study_xgb = optuna.create_study(direction="minimize",
                                sampler=optuna.samplers.TPESampler(seed=SEED))
study_xgb.optimize(objective_xgb, n_trials=80, show_progress_bar=True)
best_xgb = study_xgb.best_params
print(f"\n  Best CV MAE (XGB): {study_xgb.best_value:,.2f}")

# ============================================================
# 7. OPTUNA — LGB
# ============================================================
print("\n" + "=" * 60)
print("7. OPTUNA — LGB REVENUE")
print("=" * 60)

def objective_lgb(trial):
    params = {
        "n_estimators":     trial.suggest_int("n_estimators", 800, 5000, step=100),
        "learning_rate":    trial.suggest_float("learning_rate", 0.003, 0.04, log=True),
        "max_depth":        trial.suggest_int("max_depth", 3, 8),
        "num_leaves":       trial.suggest_int("num_leaves", 15, 100),
        "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-3, 50.0, log=True),
        "reg_alpha":        trial.suggest_float("reg_alpha",  0.01, 20.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 0.05, 20.0, log=True),
        "subsample_freq":   trial.suggest_int("subsample_freq", 1, 10),
        "min_split_gain":   trial.suggest_float("min_split_gain", 0.0, 1.0),
    }
    return cv_mae_lgb(params)

study_lgb = optuna.create_study(direction="minimize",
                                sampler=optuna.samplers.TPESampler(seed=SEED))
study_lgb.optimize(objective_lgb, n_trials=60, show_progress_bar=True)
best_lgb = study_lgb.best_params
print(f"\n  Best CV MAE (LGB): {study_lgb.best_value:,.2f}")

# ============================================================
# 8. OPTUNA — COGS
# ============================================================
print("\n" + "=" * 60)
print("8. OPTUNA — COGS MODEL")
print("=" * 60)

def objective_cogs(trial):
    params = {
        "n_estimators":     trial.suggest_int("n_estimators", 300, 3000, step=100),
        "learning_rate":    trial.suggest_float("learning_rate", 0.005, 0.05, log=True),
        "max_depth":        trial.suggest_int("max_depth", 3, 6),
        "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 5, 50),
        "reg_alpha":        trial.suggest_float("reg_alpha",  0.01, 5.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 0.1,  10.0, log=True),
    }
    return cv_mae_xgb(params, COGS_TGT)

study_cogs = optuna.create_study(direction="minimize",
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
study_cogs.optimize(objective_cogs, n_trials=40, show_progress_bar=True)
best_cogs = study_cogs.best_params
print(f"\n  Best CV MAE (COGS): {study_cogs.best_value:,.2f}")

# ============================================================
# 9. TRAIN BASE MODELS (train → eval on val)
# ============================================================
print("\n" + "=" * 60)
print("9. TRAINING BASE MODELS")
print("=" * 60)

# ── XGB ───────────────────────────────────────────────────────
xgb_rev = xgb.XGBRegressor(
    **best_xgb, random_state=SEED, n_jobs=-1, verbosity=0,
    eval_metric=["mae","rmse"], early_stopping_rounds=80)
xgb_rev.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=200)

# ── LGB ───────────────────────────────────────────────────────
lgb_rev = lgb.LGBMRegressor(**best_lgb, random_state=SEED, n_jobs=-1, verbose=-1)
lgb_rev.fit(X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(80, verbose=False), lgb.log_evaluation(-1)])

# ── HistGradientBoosting (sklearn, native NaN support) ────────
hgb_rev = HistGradientBoostingRegressor(
    max_iter=2000,
    learning_rate=0.02,
    max_depth=6,
    min_samples_leaf=20,
    l2_regularization=1.0,
    max_bins=255,
    early_stopping=True,
    validation_fraction=None,   # we handle val manually
    n_iter_no_change=80,
    random_state=SEED,
)
# HGB doesn't support separate eval_set — train on train, monitor nothing
# (it will use internal OOB-like stopping)
hgb_rev.fit(X_train.values, y_train.values)

# ── ExtraTrees (fast, high-variance diversity for ensemble) ───
# Use top-K features from XGB to keep ET fast & relevant
TOP_K = 80
fi_xgb  = pd.Series(xgb_rev.feature_importances_, index=FEATURES)
top_feat = fi_xgb.nlargest(TOP_K).index.tolist()

et_rev = ExtraTreesRegressor(
    n_estimators=500,
    max_depth=12,
    min_samples_leaf=5,
    max_features=0.4,
    random_state=SEED,
    n_jobs=-1,
)
et_rev.fit(X_train[top_feat].fillna(0), y_train)

# ── COGS XGB ──────────────────────────────────────────────────
xgb_cogs = xgb.XGBRegressor(
    **best_cogs, random_state=SEED, n_jobs=-1, verbosity=0,
    eval_metric="mae", early_stopping_rounds=60)
xgb_cogs.fit(X_train, y_train_cogs, eval_set=[(X_val, y_val_cogs)], verbose=False)

# ── Val OOF predictions ───────────────────────────────────────
val_xgb = np.clip(xgb_rev.predict(X_val),             0, None)
val_lgb = np.clip(lgb_rev.predict(X_val),             0, None)
val_hgb = np.clip(hgb_rev.predict(X_val.values),      0, None)
val_et  = np.clip(et_rev.predict(X_val[top_feat].fillna(0)), 0, None)

for name, pred in [("XGB",val_xgb),("LGB",val_lgb),
                   ("HGB",val_hgb),("ET", val_et)]:
    print(f"  [{name}] Val MAE={mean_absolute_error(y_val,pred):,.0f}  "
          f"R²={r2_score(y_val,pred):.4f}")

# ============================================================
# 10. STACKING META-LEARNER (Ridge)
# ============================================================
print("\n" + "=" * 60)
print("10. STACKING META-LEARNER")
print("=" * 60)

oof_val   = np.column_stack([val_xgb, val_lgb, val_hgb, val_et])
scaler    = StandardScaler()
oof_val_s = scaler.fit_transform(oof_val)

meta = Ridge(alpha=1.0, positive=True)
meta.fit(oof_val_s, y_val)
val_meta = np.clip(meta.predict(oof_val_s), 0, None)
print(f"  [META] Val MAE={mean_absolute_error(y_val,val_meta):,.0f}  "
      f"R²={r2_score(y_val,val_meta):.4f}")
print(f"  Weights → XGB:{meta.coef_[0]:.3f}  LGB:{meta.coef_[1]:.3f}  "
      f"HGB:{meta.coef_[2]:.3f}  ET:{meta.coef_[3]:.3f}")

val_pred = val_meta

# ── Retrain all on train+val ──────────────────────────────────
print("\n  Retraining on full train+val...")

xgb_rev_full = xgb.XGBRegressor(
    **best_xgb, random_state=SEED, n_jobs=-1, verbosity=0)
xgb_rev_full.fit(X_trainval, y_trainval, verbose=False)

lgb_rev_full = lgb.LGBMRegressor(
    **{**best_lgb, "n_estimators": lgb_rev.best_iteration_ + 80},
    random_state=SEED, n_jobs=-1, verbose=-1)
lgb_rev_full.fit(X_trainval, y_trainval)

hgb_rev_full = HistGradientBoostingRegressor(
    max_iter=hgb_rev.n_iter_ + 50,
    learning_rate=0.02, max_depth=6,
    min_samples_leaf=20, l2_regularization=1.0,
    max_bins=255, random_state=SEED)
hgb_rev_full.fit(X_trainval.values, y_trainval.values)

et_rev_full = ExtraTreesRegressor(
    n_estimators=500, max_depth=12, min_samples_leaf=5,
    max_features=0.4, random_state=SEED, n_jobs=-1)
et_rev_full.fit(X_trainval[top_feat].fillna(0), y_trainval)

xgb_cogs_full = xgb.XGBRegressor(
    **best_cogs, random_state=SEED, n_jobs=-1, verbosity=0)
xgb_cogs_full.fit(X_trainval, y_trainval_cogs, verbose=False)

# ============================================================
# 11. SEASONAL NAIVE BASELINE
# ============================================================
print("\n" + "=" * 60)
print("11. SEASONAL NAIVE BASELINE")
print("=" * 60)

train_sales = sales.copy()
train_sales["month"]     = train_sales["Date"].dt.month
train_sales["day"]       = train_sales["Date"].dt.day
train_sales["dayofweek"] = train_sales["Date"].dt.dayofweek

test_dates_list = pd.date_range("2023-01-01", sample_sub["Date"].max(), freq="D")

naive_preds = []
for dt in test_dates_list:
    same_md = train_sales[
        (train_sales["month"] == dt.month) &
        (train_sales["day"]   == dt.day)]["Revenue"]
    if len(same_md) == 0:
        same_md = train_sales[
            (train_sales["month"]     == dt.month) &
            (train_sales["dayofweek"] == dt.dayofweek())]["Revenue"]
    naive_preds.append(same_md.tail(3).mean() if len(same_md) > 0 else np.nan)

naive_preds = np.array(naive_preds)
for i in range(len(naive_preds)):
    if np.isnan(naive_preds[i]) and i > 0:
        naive_preds[i] = naive_preds[i - 1]
print(f"  Naive range: {np.nanmin(naive_preds):,.0f} → {np.nanmax(naive_preds):,.0f}")

# ============================================================
# 12. RECURSIVE FORECASTING + DRIFT CORRECTION
# ============================================================
print("\n" + "=" * 60)
print("12. RECURSIVE FORECASTING + DRIFT CORRECTION")
print("=" * 60)

n_total = len(base)
n_known = len(sales)
running_rev  = np.full(n_total, np.nan)
running_cogs = np.full(n_total, np.nan)
running_rev[:n_known]  = sales["Revenue"].values
running_cogs[:n_known] = sales["COGS"].values

date_to_pos = {row["Date"]: idx for idx, row in base.iterrows()}

test_preds_xgb  = []
test_preds_lgb  = []
test_preds_hgb  = []
test_preds_et   = []
test_preds_cogs = []

MAX_LAG = max(LAGS) + 10
print(f"  Forecasting {len(test_dates_list)} days recursively...")

for i, forecast_date in enumerate(test_dates_list):
    pos = date_to_pos.get(pd.Timestamp(forecast_date))
    if pos is None:
        for lst in [test_preds_xgb, test_preds_lgb,
                    test_preds_hgb, test_preds_et, test_preds_cogs]:
            lst.append(np.nan)
        continue

    start = max(0, pos - MAX_LAG)
    temp_base = base.iloc[start:pos+1].copy().reset_index(drop=True)
    temp_rev  = pd.Series(running_rev[start:pos+1])
    temp_cogs = pd.Series(running_cogs[start:pos+1])

    temp_df = add_lag_features(temp_base, temp_rev, temp_cogs)
    temp_df["sessions_x_promo"]    = temp_df["sessions_lag1"]        * temp_df["promo_flag"]
    temp_df["traffic_x_discount"]  = temp_df["sessions_lag1"]        * temp_df["max_discount"]
    temp_df["visitors_x_promo"]    = temp_df["unique_visitors_lag1"] * temp_df["promo_flag"]

    row     = temp_df.iloc[[-1]][FEATURES].fillna(0)
    row_top = temp_df.iloc[[-1]][top_feat].fillna(0)

    p_xgb  = float(np.clip(xgb_rev_full.predict(row),              0, None)[0])
    p_lgb  = float(np.clip(lgb_rev_full.predict(row),              0, None)[0])
    p_hgb  = float(np.clip(hgb_rev_full.predict(row.values),       0, None)[0])
    p_et   = float(np.clip(et_rev_full.predict(row_top),           0, None)[0])
    p_cogs = float(np.clip(xgb_cogs_full.predict(row),             0, None)[0])

    test_preds_xgb.append(p_xgb)
    test_preds_lgb.append(p_lgb)
    test_preds_hgb.append(p_hgb)
    test_preds_et.append(p_et)
    test_preds_cogs.append(p_cogs)

    # Meta-blend for feedback into running series
    stack_row      = scaler.transform([[p_xgb, p_lgb, p_hgb, p_et]])
    p_meta         = float(np.clip(meta.predict(stack_row), 0, None)[0])
    running_rev[pos]  = p_meta
    running_cogs[pos] = p_cogs

    if (i + 1) % 50 == 0:
        print(f"    Day {i+1}/{len(test_dates_list)} — "
              f"{forecast_date.date()} → {p_meta:,.0f}")

test_preds_xgb  = np.array(test_preds_xgb)
test_preds_lgb  = np.array(test_preds_lgb)
test_preds_hgb  = np.array(test_preds_hgb)
test_preds_et   = np.array(test_preds_et)
test_preds_cogs = np.array(test_preds_cogs)

# ── Stack ─────────────────────────────────────────────────────
stack_test  = scaler.transform(
    np.column_stack([test_preds_xgb, test_preds_lgb,
                     test_preds_hgb, test_preds_et]))
model_preds = np.clip(meta.predict(stack_test), 0, None)

# ── Drift correction (every 30 days) ─────────────────────────
DRIFT_WINDOW = 30
corrected = model_preds.copy()
for s in range(0, len(corrected), DRIFT_WINDOW):
    e = min(s + DRIFT_WINDOW, len(corrected))
    seg_model = corrected[s:e].mean()
    seg_naive = naive_preds[s:e].mean()
    if seg_model > 0 and seg_naive > 0:
        ratio = seg_model / seg_naive
        if ratio > 1.4:
            corrected[s:e] *= (1.0 / ratio) ** 0.3
        elif ratio < 0.6:
            corrected[s:e] *= (1.0 / ratio) ** 0.3

# ── Adaptive naive blend ──────────────────────────────────────
n_days  = len(test_dates_list)
model_w = np.clip(0.85 - 0.0003 * np.arange(n_days), 0.60, 0.85)
naive_w = 1 - model_w

final_preds = model_w * corrected + naive_w * naive_preds
final_preds = np.clip(final_preds, 0, None)

# ── Outlier clip ──────────────────────────────────────────────
p1  = np.nanpercentile(sales["Revenue"], 1)
p99 = np.nanpercentile(sales["Revenue"], 99) * 1.5
final_preds = np.clip(final_preds, p1, p99)

print(f"\n  Final pred range: {np.nanmin(final_preds):,.0f} → {np.nanmax(final_preds):,.0f}")

# ============================================================
# 13. SEASONALITY ANALYSIS
# ============================================================
print("\n" + "=" * 60)
print("13. SEASONALITY ANALYSIS")
print("=" * 60)

month_names = ["Jan","Feb","Mar","Apr","May","Jun",
               "Jul","Aug","Sep","Oct","Nov","Dec"]
dow_names   = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]

m_idx = train_hist.groupby("month")["Revenue"].mean()
m_idx = m_idx / m_idx.mean()
q_idx = train_hist.groupby("quarter")["Revenue"].mean()
q_idx = q_idx / q_idx.mean()
d_idx = train_hist.groupby("dayofweek")["Revenue"].mean()
d_idx = d_idx / d_idx.mean()

print("\n  Monthly Seasonality Index:")
for m, idx in m_idx.items():
    bar = "█" * int(idx * 12)
    tag = " ← PEAK"   if idx == m_idx.max() else \
          " ← TROUGH" if idx == m_idx.min() else ""
    print(f"    {month_names[m-1]:3s} M{m:02d}: {idx:.3f}  {bar}{tag}")

print("\n  Quarterly Seasonality Index:")
for q, idx in q_idx.items():
    print(f"    Q{q}: {idx:.3f}  {'█' * int(idx * 12)}")

yoy = train_hist.groupby("year")["Revenue"].sum()
print("\n  Annual Revenue (YoY):")
prev = None
for yr, rev in yoy.items():
    growth = f"  ({(rev/prev - 1)*100:+.1f}% YoY)" if prev else ""
    tag    = " ← slowdown begins" if yr == 2019 else ""
    print(f"    {yr}: {rev:>15,.0f}{growth}{tag}")
    prev = rev

pd.DataFrame({
    "month": range(1, 13),
    "month_name": month_names,
    "seasonality_index": [m_idx.get(m, 1.0) for m in range(1, 13)],
}).to_csv("seasonality_index.csv", index=False)

# ============================================================
# 14. FEATURE IMPORTANCE
# ============================================================
print("\n" + "=" * 60)
print("14. FEATURE IMPORTANCE (Top 30 — XGB)")
print("=" * 60)

fi = (pd.DataFrame({"feature": FEATURES,
                    "importance": xgb_rev.feature_importances_})
      .sort_values("importance", ascending=False)
      .head(30))
print(fi.to_string(index=False))
fi.to_csv("feature_importance.csv", index=False)

# ============================================================
# 15. SUBMISSION
# ============================================================
print("\n" + "=" * 60)
print("15. GENERATING SUBMISSION")
print("=" * 60)

submission = sample_sub.copy()
pred_map   = dict(zip(pd.to_datetime(test_dates_list), final_preds))
cogs_map   = dict(zip(pd.to_datetime(test_dates_list), test_preds_cogs))

submission["Revenue"] = pd.to_datetime(submission["Date"]).map(pred_map).round(2)
if "COGS" in submission.columns:
    cogs_ratio_final = (sales["COGS"].tail(90) / (sales["Revenue"].tail(90) + 1)).mean()
    model_cogs  = pd.to_datetime(submission["Date"]).map(cogs_map)
    naive_cogs  = submission["Revenue"] * cogs_ratio_final
    submission["COGS"] = (0.7 * model_cogs + 0.3 * naive_cogs).round(2)

submission["Date"] = pd.to_datetime(submission["Date"]).dt.strftime("%Y-%m-%d")
submission.to_csv("submission.csv", index=False)

print(f"  Rows          : {len(submission)}")
print(f"  Revenue range : {submission['Revenue'].min():.2f} → {submission['Revenue'].max():.2f}")
print("  submission.csv ✓")

mae_v  = mean_absolute_error(y_val, val_pred)
rmse_v = np.sqrt(mean_squared_error(y_val, val_pred))
r2_v   = r2_score(y_val, val_pred)
pd.DataFrame([{
    "Val_MAE":          mae_v,
    "Val_RMSE":         rmse_v,
    "Val_R2":           r2_v,
    "XGB_CV_MAE":       study_xgb.best_value,
    "LGB_CV_MAE":       study_lgb.best_value,
    "COGS_CV_MAE":      study_cogs.best_value,
    "Meta_w_XGB":       meta.coef_[0],
    "Meta_w_LGB":       meta.coef_[1],
    "Meta_w_HGB":       meta.coef_[2],
    "Meta_w_ET":        meta.coef_[3],
}]).to_csv("validation_metrics.csv", index=False)
print("  validation_metrics.csv ✓")

print("\n" + "=" * 60)
print("DONE ✓")
print("=" * 60)

1. LOADING DATA
  Train: 2012-07-04 → 2022-12-31 (3833 rows)
  Test : 2023-01-01 → 2024-07-01 (548 rows)

2. STATIC FEATURE ENGINEERING
  Building promotions features...
  Building web traffic features (lagged)...
  Building inventory features (lagged monthly)...
  Static features built: 99 columns
  Building lag/rolling features...
  Total features: 201 columns

4. TRAIN / VAL / TEST SPLIT
  Train   : 2012-07-04 → 2021-12-31 (3468 rows)
  Val     : 2022-01-01 → 2022-12-31 (365 rows)
  Test    : 2023-01-01 → 2024-07-01 (548 rows)
  Features: 198

6. OPTUNA — XGB REVENUE


  0%|          | 0/80 [00:00<?, ?it/s]


  Best CV MAE (XGB): 476,076.58

7. OPTUNA — LGB REVENUE


  0%|          | 0/60 [00:00<?, ?it/s]


  Best CV MAE (LGB): 480,787.99

8. OPTUNA — COGS MODEL


  0%|          | 0/40 [00:00<?, ?it/s]


  Best CV MAE (COGS): 419,864.53

9. TRAINING BASE MODELS
[0]	validation_0-mae:1703718.46712	validation_0-rmse:2009547.07824
[200]	validation_0-mae:544024.59812	validation_0-rmse:758932.50960
[400]	validation_0-mae:529513.42055	validation_0-rmse:729633.30836
[600]	validation_0-mae:522597.91404	validation_0-rmse:717477.22214
[800]	validation_0-mae:518655.02140	validation_0-rmse:710709.61384
[988]	validation_0-mae:518868.26387	validation_0-rmse:709809.33464
  [XGB] Val MAE=518,177  R²=0.8208
  [LGB] Val MAE=534,329  R²=0.8171
  [HGB] Val MAE=547,583  R²=0.8113
  [ET] Val MAE=554,494  R²=0.7833

10. STACKING META-LEARNER
  [META] Val MAE=510,957  R²=0.8279
  Weights → XGB:936661.844  LGB:492275.612  HGB:94420.905  ET:0.000

  Retraining on full train+val...

11. SEASONAL NAIVE BASELINE
  Naive range: 623,020 → 10,366,756

12. RECURSIVE FORECASTING + DRIFT CORRECTION
  Forecasting 548 days recursively...
    Day 50/548 — 2023-02-19 → 4,227,110
    Day 100/548 — 2023-04-10 → 4,900,641
    